In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/Enhanced_IDS_Project"

os.makedirs(project_path, exist_ok=True)
os.makedirs(project_path + "/processed_data", exist_ok=True)

print("Project folder created!")
print(project_path)

Project folder created!
/content/drive/MyDrive/Enhanced_IDS_Project


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder

import os
import joblib

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving KDDTrain+.csv to KDDTrain+.csv
Saving KDDTest+.csv to KDDTest+.csv


In [ ]:
import os

print(os.listdir())

['.config', 'KDDTrain+.csv', 'KDDTest+.csv', 'sample_data']


In [ ]:
train_path = "KDDTrain+.csv"
test_path = "KDDTest+.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training data loaded successfully!")
print("Testing data loaded successfully!")

Training data loaded successfully!
Testing data loaded successfully!


In [ ]:
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (125973, 43)
Testing shape: (22543, 43)


In [ ]:
display(train_df.head())

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [ ]:
display(test_df.head())

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal,21
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint,15
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11


In [ ]:
print("Columns:")
for i, column in enumerate(train_df.columns):
    print(i, ":", column)

Columns:
0 : duration
1 : protocol_type
2 : service
3 : flag
4 : src_bytes
5 : dst_bytes
6 : land
7 : wrong_fragment
8 : urgent
9 : hot
10 : num_failed_logins
11 : logged_in
12 : num_compromised
13 : root_shell
14 : su_attempted
15 : num_root
16 : num_file_creations
17 : num_shells
18 : num_access_files
19 : num_outbound_cmds
20 : is_host_login
21 : is_guest_login
22 : count
23 : srv_count
24 : serror_rate
25 : srv_serror_rate
26 : rerror_rate
27 : srv_rerror_rate
28 : same_srv_rate
29 : diff_srv_rate
30 : srv_diff_host_rate
31 : dst_host_count
32 : dst_host_srv_count
33 : dst_host_same_srv_rate
34 : dst_host_diff_srv_rate
35 : dst_host_same_src_port_rate
36 : dst_host_srv_diff_host_rate
37 : dst_host_serror_rate
38 : dst_host_srv_serror_rate
39 : dst_host_rerror_rate
40 : dst_host_srv_rerror_rate
41 : label
42 : difficulty


In [ ]:
print("Training labels:")
print(train_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())

Training labels:
label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

Testing labels:
label
normal             9710
neptune            4657
guess_passwd       1231
mscan               996
warezmaster         944
apache2             737
satan               735
processtable        685
smurf               665
back                359
snmpguess           331
saint               319
mailbomb            293
snmpgetattack       178
portsweep        

In [ ]:
train_df = train_df.drop(columns=["difficulty"])
test_df = test_df.drop(columns=["difficulty"])

print("Difficulty column removed.")
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Difficulty column removed.
Training shape: (125973, 42)
Testing shape: (22543, 42)


In [ ]:
train_df["label"] = train_df["label"].apply(
    lambda x: 0 if x == "normal" else 1
)

test_df["label"] = test_df["label"].apply(
    lambda x: 0 if x == "normal" else 1
)

print("Labels converted successfully!")

Labels converted successfully!


In [ ]:
print("Training labels:")
print(train_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())

Training labels:
label
0    67343
1    58630
Name: count, dtype: int64

Testing labels:
label
1    12833
0     9710
Name: count, dtype: int64


In [ ]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (125973, 41)
y_train shape: (125973,)
X_test shape: (22543, 41)
y_test shape: (22543,)


In [ ]:
categorical_columns = [
    "protocol_type",
    "service",
    "flag"
]

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['protocol_type', 'service', 'flag']


In [ ]:
numerical_columns = [
    column for column in X_train.columns
    if column not in categorical_columns
]

print("Number of numerical features:", len(numerical_columns))
print("Number of categorical features:", len(categorical_columns))

Number of numerical features: 38
Number of categorical features: 3


In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(
    X_train[categorical_columns]
)

X_test_cat = encoder.transform(
    X_test[categorical_columns]
)

print("Categorical encoding completed!")
print("Encoded training shape:", X_train_cat.shape)
print("Encoded testing shape:", X_test_cat.shape)

Categorical encoding completed!
Encoded training shape: (125973, 84)
Encoded testing shape: (22543, 84)


In [ ]:
X_train_num = X_train[numerical_columns].to_numpy()
X_test_num = X_test[numerical_columns].to_numpy()

X_train_encoded = np.hstack([
    X_train_num,
    X_train_cat
])

X_test_encoded = np.hstack([
    X_test_num,
    X_test_cat
])

print("Final encoded training shape:", X_train_encoded.shape)
print("Final encoded testing shape:", X_test_encoded.shape)

Final encoded training shape: (125973, 122)
Final encoded testing shape: (22543, 122)


In [ ]:
print("Original train rows:", len(train_df))
print("Original test rows:", len(test_df))

print("X_train rows:", len(X_train))
print("X_test rows:", len(X_test))

print("Encoded train rows:", len(X_train_encoded))
print("Encoded test rows:", len(X_test_encoded))

Original train rows: 125973
Original test rows: 22543
X_train rows: 125973
X_test rows: 22543
Encoded train rows: 125973
Encoded test rows: 22543


In [ ]:
train_mean = np.mean(X_train_encoded, axis=0)

train_min = np.min(X_train_encoded, axis=0)
train_max = np.max(X_train_encoded, axis=0)

train_range = train_max - train_min

# Avoid division by zero
train_range[train_range == 0] = 1

X_train_normalized = (
    X_train_encoded - train_mean
) / train_range

X_test_normalized = (
    X_test_encoded - train_mean
) / train_range

print("Mean normalization completed!")
print("Normalized training shape:", X_train_normalized.shape)
print("Normalized testing shape:", X_test_normalized.shape)

Mean normalization completed!
Normalized training shape: (125973, 122)
Normalized testing shape: (22543, 122)


In [ ]:
print("Minimum value:", X_train_normalized.min())
print("Maximum value:", X_train_normalized.max())

print("\nFirst 5 rows:")
print(X_train_normalized[:5])

Minimum value: -0.815166742079652
Maximum value: 0.9999920617910187

First 5 rows:
[[-6.69210054e-03 -3.26644367e-05 -1.50992822e-05 -1.98455225e-04
  -7.56246709e-03 -3.70449752e-05 -2.65466080e-03 -2.44496837e-04
  -3.95735594e-01 -3.73379216e-05 -1.34155732e-03 -5.51705524e-04
  -4.04648821e-05 -2.94636780e-04 -2.06393434e-04 -4.55123982e-04
   0.00000000e+00 -7.93820898e-06 -9.42265406e-03 -1.60680146e-01
  -5.03676874e-02 -2.84484532e-01 -2.82485374e-01 -1.19958483e-01
  -1.21183269e-01  3.39072341e-01 -6.30526383e-02 -9.73216483e-02
  -1.26074293e-01 -3.55501980e-01 -3.51241695e-01 -5.29511086e-02
   2.16211410e-02 -3.25424496e-02 -2.84452462e-01 -2.78484517e-01
  -6.88318132e-02 -1.20239893e-01 -6.58156907e-02  1.84833258e-01
  -1.19017567e-01 -1.48444508e-03 -5.79489256e-04 -6.84273614e-03
  -1.58764180e-05 -7.58098958e-03 -5.63612838e-03 -5.82664539e-03
  -4.32632389e-03 -4.46921166e-03 -4.13580688e-03 -4.27075643e-03
  -4.51684091e-03 -7.17852238e-02 -3.44518270e-03 -3.640462

In [ ]:
print("NaN values in training:",
      np.isnan(X_train_normalized).sum())

print("NaN values in testing:",
      np.isnan(X_test_normalized).sum())

print("Infinite values in training:",
      np.isinf(X_train_normalized).sum())

print("Infinite values in testing:",
      np.isinf(X_test_normalized).sum())

NaN values in training: 0
NaN values in testing: 0
Infinite values in training: 0
Infinite values in testing: 0


In [ ]:
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

y_train shape: (125973,)
y_test shape: (22543,)


In [ ]:
import os

project_path = "/content/drive/MyDrive/Enhanced_IDS_Project"

os.makedirs(project_path, exist_ok=True)
os.makedirs(project_path + "/processed_data", exist_ok=True)

print("Project folder ready!")

Project folder ready!


In [ ]:
np.save(
    project_path + "/processed_data/X_train_normalized.npy",
    X_train_normalized
)

np.save(
    project_path + "/processed_data/X_test_normalized.npy",
    X_test_normalized
)

np.save(
    project_path + "/processed_data/y_train.npy",
    y_train
)

np.save(
    project_path + "/processed_data/y_test.npy",
    y_test
)

joblib.dump(
    encoder,
    project_path + "/processed_data/encoder.pkl"
)

np.save(
    project_path + "/processed_data/train_mean.npy",
    train_mean
)

np.save(
    project_path + "/processed_data/train_range.npy",
    train_range
)

print("All processed files saved to Google Drive!")

All processed files saved to Google Drive!


In [ ]:
print("Saved files:")

for file in os.listdir(project_path + "/processed_data"):
    print(file)

Saved files:
X_train_normalized.npy
X_test_normalized.npy
y_train.npy
y_test.npy
encoder.pkl
train_mean.npy
train_range.npy
